## MNIST met Linear Regression

Bij K-means onthielden we gemiddelde plaatjes.
Nu doen we iets anders:

- We leren per cijfer een formule die zegt hoe waarschijnlijk het is dat een plaatje dat cijfer is.

Dat klinkt spannend, maar het komt neer op:

- pixels krijgen een gewicht

- sommige pixels tellen zwaarder mee dan andere

In [62]:
# imports
import numpy as np;
# from LinearRegression_methods import *;
from tensorflow.keras.datasets import mnist;
import random as rnd;

In [39]:
# get data
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

In [45]:
from sklearn.linear_model import LinearRegression


def normalize_image(image: np.array):
    """
    normalize values between 0 and 1
    """

    maximum = np.max(image)
    normalised: np.array = image / maximum

    return normalised


def normalize_and_vectorize_image(image: np.array):
    """
    Turn a image of variable dimesions (must be square) into a vector and normalize values between 0 and 1
    """
    flat = image.flatten()

    maximum = np.max(flat)
    normalised: np.array = flat / maximum
    return normalised


def train_linear_model(X_train, y_binary):
    model = LinearRegression()
    model.fit(X_train, y_binary)
    print(f"coef {model.coef_.shape}")
    return model


def train_linear_model(X_train, y_binary):
    model = LinearRegression()
    model.fit(X_train, y_binary)
    return model

def determine_accuracy(prototypes, images, labels):
    total = len(images)
    misses = 0
    miss_for_num: dict  = {0: 0, 1: 0, 2:0, 3:0, 4:0, 5:0, 6:0, 7:0, 8:0, 9:0}
    for i, image in enumerate(images):
        prediction = predict(prototypes, image)
        # print(f"pred: {prediction} | act: {labels[i]}")
        if prediction != labels[i]:
            misses += 1
            miss_for_num[labels[i]] += 1
    print(f"misses {misses}")
    print(f"accuracy {(total - misses) / total * 100} %")
    print(miss_for_num)


def predict(models, image):
    results = {}
    for i in models:
        results[i] = models[i].predict([image])
    best_fit = max(results, key= lambda k: results[k])
    return best_fit

In [ ]:
print(0)

0


In [46]:
# preprocess data
train_images_vectored = np.array([normalize_and_vectorize_image(img) for img in train_images])
test_images_vectored = np.array([normalize_and_vectorize_image(img) for img in test_images])
print("finished")


finished


### Laden MNIST

Je mag je code uit de Kmeans hergebruiken waar van toepassing.

Schrijf een functie die:

- MNIST laadt

- pixelwaarden schaalt naar 0–1

- labels omzet naar gehele getallen

**Vragen:**

Waarom schalen we pixelwaarden?

Hoeveel features heeft één afbeelding?

__antwoorden__

Door het schalen van de pixelwaarde wordt het eenvoudiger te vergelijken.\
Want dankzij het schalen krijgt iedere afbeelding een pixel met waarde 0 en een pixel met waarde 1. Als dit niet wordt gedaan is er de mogelijkheid dat een afbeelding in het geheel donkerder is en dus een hoogste pixelwaarde van 0.5 heeft.\

Iedere afbeelding heeft 28 * 28 = 784 features

### Labels geschikt maken voor regressie

Linear regression werkt met getallen, niet met categorieën.

Waarom is dat?

We doen daarom one-vs-rest:

- eg voor cijfer 3:

- alle 3’en → 1

- alle andere cijfers → 0

Schrijf een functie die:

- één gekozen cijfer omzet naar binaire labels, dus 
    - 1 voor het gekozen cijfer
    - 0 voor alle andere cijfers

Vragen:

Waarom kunnen we niet direct 0–9 voorspellen?

Wat betekent een voorspelling van 0.8?

__antwoorden__

Linear regression werkt met getallen en niet met categorieën omdat linear regression een linear verband met maximale voorspellingskracht probeert op te stellen. Je kunt geen formule opstellen die met categorieën werkt.

functie is one_hot_encode_digit()

Omdat 0-9 in dit geval categorieën zijn.\
Een afbeelding van 1 + een afbeelding van 4 != afbeelding van 5.

### Lineair model trainen

We leren een gewicht per pixel.

Intuïtief:

- heldere pixels op logische plekken krijgen hoge gewichten

- achtergrond krijgt lage gewichten


Je mag de volgende functie gebruiken:

```python
def train_linear_model(X_train, y_binary):
    model = LinearRegression()
    model.fit(X_train, y_binary)
    return model
```
Kort samengevat: Linearie Regressie berekent exact de optimale waarden voor **intercept** & **coefficients**.

Vragen:

- Wat is X_train hier?

- En y_binary?

- Wat stelt één gewicht voor?

- Hoeveel gewichten heeft het model? Tip: kijk naar **model.coef_**

- Wat is model.intercept_ denk je?

__antwoorden__

- X_train zijn afbeeldingen
- y_binary zijn de labels na encoding
- Het gewicht stelt het belang van een pixel voor, voor de uiteindelijke voorspelling.
- Het mode heeft 784 weights. Een voor iedere pixel dus.
- De intercept kan worden gebruikt bij data die niet gecentreerd is om linear regression er toch op te kunnen laten werken

In [47]:
from sklearn.linear_model import LinearRegression

def train_linear_model(X_train, y_binary) -> LinearRegression:
    model = LinearRegression()
    model.fit(X_train, y_binary)
    return model



def train_numbers(train_images, train_labels):
    models = {}
    for num in range(10):
        models[num] = train_linear_model(train_images, [label == num for label in train_labels])
        print(f"finished for {num}")
    return models




In [73]:
model_one_image = train_numbers([train_images_vectored[0]], [train_labels[0]])
model = train_linear_model(train_images_vectored, [label == 0 for label in train_labels])
models = train_numbers(train_images_vectored, train_labels)

finished for 0
finished for 1
finished for 2
finished for 3
finished for 4
finished for 5
finished for 6
finished for 7
finished for 8
finished for 9
finished for 0
finished for 1
finished for 2
finished for 3
finished for 4
finished for 5
finished for 6
finished for 7
finished for 8
finished for 9


### Model trainen voor 1 cijfer

Laten we eerst het model trainen voor een cijfer. 
Stop letterlijk alle 784 (genormaliseerde) pixelwaardes als features in het model
Schrijf code die dat doet

Er wordt altijd een 5 voorspelt.

In [108]:
# loaded mnist is een tuple die komt van

# def single_experiment(digit, loaded_mnist):
#     return train_linear_model(loaded_mnist, digit)

index = rnd.randint(0, 10_000)
print(f"trained for {train_labels[0]}")
print(test_labels[index])
predict(model_one_image, test_images_vectored[5])

trained for 5
2


5

### Een cijfer die we getraind hebben voorspellen

Voorspel nu of een test cijfer inderdaad is wat je verwacht.

Maak een functie predict_num met de volgende argumenten:
- model
- image
- echte image waarde

Leg uit waarom we deze 3 nodig hebben?

Je kunt voor het voorspellen de volgende functie gebruiken:
```python
    score = model.predict([image])[0]
```

Waarom is die [0] er? zoek het uit

Score is afhankelijk van het model. In dit geval kun je het lezen als:

- Score > 0 → model denkt: dit is waarschijnlijk een 5

- Score < 0 → model denkt: dit is geen 5

- |score| groot → model is zekerder

__antwoorden__

Deze drie waarden zijn nodig omdat:
je een afbeelding wilt voorspellen. Je hebt dus de afbeelding nodig evenals het model waarmee de voorspelling wordt gemaakt. De echte image waarde (label) is nodig om te controleren of dit klopt.

Met model.predict([image])[0] wordt de voorspelling dat een nummer 0 is gegeven.

In [59]:
# def predict_num(model, image):
#     #print(model.predict([image]))
#     return model.predict([image])
#     #return score



# index = 10
# print(test_labels[index])
# prediction = predict_num(model, test_images[index], test_labels[index])
# print(prediction)


### Experimenteren op een echt getal

Train het nu op bijvoorbeeld getal 5.

Maak nu code waarbij we een random getal uit de database halen en deze classificeren. Je mag de volgende code gebruiken om het getal te tonen:

```python
plt.imshow(random_image.reshape(28, 28), cmap="gray")
plt.title(f"Echte label: {true_label}")
plt.axis("off")
plt.show()
```


In [ ]:

print(test_labels[5])
predict(models, test_images_vectored[5])

1


1

### Modellen trainen voor alle cijfers

We trainen 10 aparte modellen:

- één voor 0

- één voor 1

- etc...

Schrijf een functie die:

- voor elk cijfer een model traint

- alle modellen opslaat in een datastructuur

### Voorspellen

Een nieuw plaatje:

- gaat door alle 10 modellen

- elk model geeft een score

- hoogste score wint

Schrijf een functie **predict_case** die:

- voor elk testplaatje voorspelt welk cijfer het is

- gebruik **model.predict([image])**

Vragen:

Wat betekent een hoge score?

Waarom kiezen we de hoogste score?

In [55]:
determine_accuracy(models, test_images_vectored, test_labels)

misses 1398
accuracy 86.02 %
{0: 36, 1: 28, 2: 219, 3: 131, 4: 101, 5: 233, 6: 83, 7: 144, 8: 215, 9: 208}


### Gebruik nu je eigen features

In [56]:
def check_center(image):
    center_size = 8
    start = (image.shape[0] - center_size) // 2
    end = start + center_size
    center_square = image[start:end, start:end]
    score = center_square.mean() / 255.0
    return score

def vertical_symmetry(image):
    flipped = np.fliplr(image)
    diff = np.abs(image - flipped)
    score = 1 - (diff.mean() / 255.0)
    return score

def horizontal_symmetry(image):
    flipped = np.flipud(image)
    diff = np.abs(image - flipped)
    score = 1 - (diff.mean() / 255.0)
    return score

def top_bottom_balance(image):
    mid = image.shape[0] // 2
    top = image[:mid, :].mean()
    bottom = image[mid:, :].mean()
    return (top - bottom) / 255.0

def pixel_density(image):
    return image.mean() / 255.0

def left_right_balance(image):
    mid = image.shape[1] // 2
    left = image[:, :mid].mean()
    right = image[:, mid:].mean()
    return (left - right) / 255.0

def active_pixels(image):
    return np.sum(image > 50) / (28 * 28)

sort_method = {
    "center": check_center,
    "vertical_sim": vertical_symmetry,
    "horizontal_sim": horizontal_symmetry,
    "top-bottom-bal": top_bottom_balance,
    "left-right-bal": left_right_balance,
    "density": pixel_density,
    "active": active_pixels,
}



def make_image_feature_vector(image):
    features = []
    for method_name in sort_method:
        method = sort_method[method_name]
        features.append(method(image))
    return np.array(features)

def make_image_feature_vectors(images):
    feature_vectors = []
    for image in images:
        feature_vectors.append(make_image_feature_vector(image))
    return np.array(feature_vectors)

feature_vectors = make_image_feature_vectors(train_images)
print("finished making feature vectors")
models_features = train_numbers(feature_vectors, train_labels)
print("finished making linear regression")
test_feature_vectors = make_image_feature_vectors(test_images)
print("predicting")
determine_accuracy(models_features, test_feature_vectors, test_labels)

finished making feature vectors
finished for 0
finished for 1
finished for 2
finished for 3
finished for 4
finished for 5
finished for 6
finished for 7
finished for 8
finished for 9
finished making linear regression
predicting
misses 5145
accuracy 48.55 %
{0: 61, 1: 18, 2: 508, 3: 784, 4: 910, 5: 815, 6: 485, 7: 436, 8: 258, 9: 870}


Gebruik een functie uit de vorige opdrachten om je eigen features te gebruiken voor linear regression.


def extract_features(img):
    features = []
    features.append(img.mean())                      # gemiddelde intensiteit
    features.append((img > 0.5).sum())               # aantal "actieve" pixels
    ...
    ....
    return features


## Rapporteer: 




De gemaakte opdrachten en de discussie/feedbackmoeten worden uitgewerkt en op CodeGrade geplaatst onder **P6 Linear Regression**